# ALand-AI Fine-tuning
Fine-tune Qwen2.5-1.5B dengan dataset bahasa Indonesia, export ke GGUF.

**Pastikan Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Install dependencies
!pip install -q transformers peft datasets accelerate bitsandbytes huggingface_hub
!apt-get install -qq git-lfs
!pip install -q llama-cpp-python

In [ ]:
# 2. Clone repo kamu (ganti URL dengan repo GitHub kamu)
GITHUB_REPO = 'https://github.com/USERNAME/AI-EVIL-BY-ALAND.git'  # ← GANTI INI
HF_TOKEN    = 'hf_...'   # ← GANTI: token dari huggingface.co/settings/tokens
HF_REPO     = 'USERNAME/aland-id'  # ← GANTI: nama repo HuggingFace kamu

!git clone {GITHUB_REPO} /content/aland
DATASET_PATH = '/content/aland/aland-ai/training/dataset.jsonl'
print('Dataset:', DATASET_PATH)

In [ ]:
# 3. Load dataset
import json
from datasets import Dataset

data = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            msgs = obj.get('messages', [])
            if len(msgs) >= 2:
                data.append({'messages': msgs})
        except: pass

import random
random.shuffle(data)
data = data[:50000]  # batasi 50k untuk Colab gratis
print(f'Dataset: {len(data)} pasangan')
dataset = Dataset.from_list(data)

In [ ]:
# 4. Load model base + tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # 1.5B — muat di Colab gratis

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True
)
print('Model loaded ✅')

In [ ]:
# 5. Setup LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Tokenize dataset
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    tok = tokenizer(text, truncation=True, max_length=512, padding='max_length')
    tok['labels'] = tok['input_ids'].copy()
    return tok

tokenized = dataset.map(format_chat, remove_columns=['messages'])
print('Tokenized ✅')

In [ ]:
# 7. Training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='/content/aland-id-lora',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()
print('Training selesai ✅')

In [ ]:
# 8. Merge LoRA + export ke GGUF
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

# Merge
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, '/content/aland-id-lora')
merged = merged.merge_and_unload()
merged.save_pretrained('/content/aland-id-merged')
tokenizer.save_pretrained('/content/aland-id-merged')
print('Merge selesai ✅')

# Convert ke GGUF (Q4_K_M)
!git clone -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q gguf
!python /content/llama.cpp/convert_hf_to_gguf.py /content/aland-id-merged \
    --outfile /content/aland-id-q4.gguf --outtype q4_k_m
print('GGUF export selesai ✅')

In [ ]:
# 9. Upload ke HuggingFace
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
api = HfApi()

# Buat repo jika belum ada
api.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)

# Upload GGUF
api.upload_file(
    path_or_fileobj='/content/aland-id-q4.gguf',
    path_in_repo='aland-id-q4.gguf',
    repo_id=HF_REPO,
)
print(f'✅ Model tersedia di: https://huggingface.co/{HF_REPO}')
print(f'\nDownload di laptop:')
print(f'  aland-ai pull {HF_REPO}')